In [1]:
import polars as pl
from glob import glob
import os

In [ ]:
# separate models
# 5k from MGB and BIDMC
# features in cum sum 3

In [ ]:
nt1_ICDs = [r'G47\.?411',r'347\.?01']
nt2_ICDs = [r'G47\.?419', r'347\.?00']
# osa_ICDs = [r'G47\.?33', r'327\.?23']
narcmeds = ['xyrem', 'xywave', 'oxybate', 'wakix', 'modafinil', 'armodafinil', 'soriamfetol', 'lumryz']
# ! NEW meds
# narcmeds = ['modafinil', 'provigil', 'armodafinil', 'nuvigil', 'solriamfetol', 'sunosi', 'pitolisant', 'wakix', 'ritalin', 'concerta', 'adderall ', 'desoxyn', 'venlafaxine', 'effexor', 'fluoxetine', 'prozac', 'duloxetine', 'cymbalta', 'drizalma sprinkle', 'sertraline', 'zoloft', 'protriptyline', 'vivactil', 'imipramine', 'clomipramine', 'anafranil', 'sodium oxybate', 'xyrem', 'lumryz', 'oxybate salts', 'xywav', 'dextroamphetamine', 'dexedrine', 'gamma-hydroxybutyrate', 'ghb', 'metadate', 'methylin', 'evekeo', 'zenzedi', 'procentra', 'paroxetine']
nt2_icd_certain = [r'G47\.?419',r'G47\.?429',r'347\.?00',r'347\.?10']
nt2_icd_maybe = [r'G47\.?10', r'G47\.?19', r'780\.?53', r'780\.?54']


for cohort in os.listdir('/home/niels/Documents/better_thunder/'):
    if cohort == 'kp':
        continue
    icd = pl.read_parquet(f'/home/niels/Documents/better_thunder/{cohort}/icd.parquet')
    med = pl.read_parquet(f'/home/niels/Documents/better_thunder/{cohort}/med.parquet').with_columns(pl.col('med').str.to_lowercase())
    demo = pl.read_parquet(f'/home/niels/Documents/better_thunder/{cohort}/demo.parquet')


    nt1_no = set(demo.filter(
        # does NOT have nt1 icd
        ~pl.col('bdsp_patient_id').is_in(
            icd.filter(
                pl.col('icd').str.contains('|'.join(nt1_ICDs))
            )['bdsp_patient_id'].unique()
        )
    ).filter(
        # does NOT have med
        ~pl.col('bdsp_patient_id').is_in(
            med.filter(
                pl.col('med').str.contains('|'.join(narcmeds))
            )['bdsp_patient_id'].unique()
        )
    )['bdsp_patient_id'])



    nt2_no = set(demo.filter(
        # does NOT have nt2 icd certain or maybe
        ~pl.col('bdsp_patient_id').is_in(
            icd.filter(
                pl.col('icd').str.contains('|'.join(nt2_icd_certain + nt2_icd_maybe))
            )['bdsp_patient_id'].unique()
        )
    ).filter(
        # does NOT have med
        ~pl.col('bdsp_patient_id').is_in(
            med.filter(
                pl.col('med').str.contains('|'.join(narcmeds))
            )['bdsp_patient_id'].unique()
        )
    )['bdsp_patient_id']).union(nt1_yes)

In [2]:
import pickle as pkl

In [8]:
mgb_count = pkl.load(open('/home/niels/cdac Dropbox/IRB_And_Admin/CONTRACTS/Takeda/Narcolepsy_NLP_Project/narco_clean/counts/mgb.pkl', 'rb'))
bidmc_count = pkl.load(open('/home/niels/cdac Dropbox/IRB_And_Admin/CONTRACTS/Takeda/Narcolepsy_NLP_Project/narco_clean/counts/bidmc.pkl', 'rb'))

In [3]:
df = pl.read_parquet('/home/niels/cdac Dropbox/IRB_And_Admin/CONTRACTS/Takeda/Narcolepsy_NLP_Project/narco_clean/annot_may_29_FINAL.parquet')

In [12]:
import numpy as np

In [33]:
rng = np.random.default_rng(42)

In [34]:
has_notes = pl.concat([pl.scan_parquet(
    x,
    parallel='prefiltered'
).select(['bdsp_patient_id']).unique() for x in glob('/home/niels/cdac Dropbox/Niels Turley/better_thunder/*/note/*.parquet')], how='diagonal_relaxed').collect()

In [35]:
mgb_count_filtered = {k: set(has_notes['bdsp_patient_id'].to_list()).intersection(v)  - set(df['bdsp_patient_id'].to_list()) for k, v in mgb_count.items()}
bidmc_count_filtered = {k: set(has_notes['bdsp_patient_id'].to_list()).intersection(v)  - set(df['bdsp_patient_id'].to_list()) for k, v in bidmc_count.items()}

In [36]:
control_pts = np.concatenate([
    rng.choice(list(mgb_count_filtered['nt1_no']), 2500, replace=False),
    rng.choice(list(mgb_count_filtered['nt2_no']), 2500, replace=False),
    rng.choice(list(bidmc_count_filtered['nt1_no']), 2500, replace=False),
    rng.choice(list(bidmc_count_filtered['nt2_no']), 2500, replace=False),
])

In [37]:
control_df = pl.DataFrame(control_pts, schema=['bdsp_patient_id']).filter(
    ~pl.col('bdsp_patient_id').is_in(df['bdsp_patient_id'].to_list())
).with_columns(
    pl.when(pl.col('bdsp_patient_id').is_in(mgb_count_filtered['nt1_no']))
    .then(pl.lit('mgb_nt1_no'))
    .when(pl.col('bdsp_patient_id').is_in(mgb_count_filtered['nt2_no']))
    .then(pl.lit('mgb_nt2_no'))
    .when(pl.col('bdsp_patient_id').is_in(bidmc_count_filtered['nt1_no']))
    .then(pl.lit('bidmc_nt1_no'))
    .otherwise(pl.lit('bidmc_nt2_no')).alias('source')
)

In [38]:
control_df

bdsp_patient_id,source
i64,str
116630638,"""mgb_nt1_no"""
113779222,"""mgb_nt1_no"""
113736323,"""mgb_nt1_no"""
120005407,"""mgb_nt1_no"""
111464494,"""mgb_nt1_no"""
…,…
150704199,"""bidmc_nt1_no"""
150748815,"""bidmc_nt1_no"""
151215187,"""bidmc_nt1_no"""


In [39]:
from prophet import Prophet
model = Prophet().model_creator.get_model('narcolepsy')

In [44]:
model.get_data_format()

{'icd': {'id': String, 'icd': String, 'date': Date},
 'med': {'id': String, 'med': String, 'date': Date},
 'note': {'id': String, 'note': String, 'date': Date}}

In [40]:
icd = pl.concat([pl.scan_parquet(
    x,
    parallel='prefiltered',
).filter(
    pl.col('bdsp_patient_id').is_in(control_df['bdsp_patient_id'].to_list())
) for x in glob('/home/niels/cdac Dropbox/Niels Turley/better_thunder/*/icd.parquet')], how='diagonal_relaxed').collect()
med = pl.concat([pl.scan_parquet(
    x,
    parallel='prefiltered',
).filter(
    pl.col('bdsp_patient_id').is_in(control_df['bdsp_patient_id'].to_list())
) for x in glob('/home/niels/cdac Dropbox/Niels Turley/better_thunder/*/med.parquet')], how='diagonal_relaxed').collect()
note = pl.concat([pl.scan_parquet(
    x,
    parallel='prefiltered',
).filter(
    pl.col('bdsp_patient_id').is_in(control_df['bdsp_patient_id'].to_list())
) for x in glob('/home/niels/cdac Dropbox/Niels Turley/better_thunder/*/note/*.parquet')], how='diagonal_relaxed').collect()

In [41]:
feat = model.preprocess(
    data = {
        'icd': icd.rename({'bdsp_patient_id':'id'}).cast({'id':pl.Utf8, 'date':pl.Date}),
        'med': med.with_columns(
            pl.col('date').str.split(' ').list.first().str.to_date('%Y-%m-%d')
        ).with_columns(
            pl.coalesce(['date', 'date_start']).cast(pl.Date).alias('date')
        ).rename({'bdsp_patient_id':'id'}).cast({'id':pl.Utf8}),
        'note': note.rename({'bdsp_patient_id':'id'}).cast({'id':pl.Utf8, 'date':pl.Date}),
    },
    show_progress=True,
)

2026-02-05 13:49:37,997	INFO worker.py:1786 -- Started a local Ray instance.
Processing narcolepsy notes:  44%|████▍     | 629500/1427086 [08:14<04:45, 2797.07it/s]Error getting result: ray::NarcolepsyProcessor.process_batch() (pid=2074359, ip=10.35.163.156, actor_id=337f12b5718a9d0c523d740c01000000, repr=<prophet.models.narcolepsy.predictor.NarcolepsyProcessor object at 0x7db56ba63bd0>)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/niels/Desktop/prophet/src/prophet/models/narcolepsy/predictor.py", line 259, in process_batch
    sentences = sent_tokenize(text)
                ^^^^^^^^^^^^^^^^^^^
  File "/home/niels/miniforge3/envs/playground/lib/python3.11/site-packages/nltk/tokenize/__init__.py", line 120, in sent_tokenize
    return tokenizer.tokenize(text)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/niels/miniforge3/envs/playground/lib/python3.11/site-packages/nltk/tokenize/punkt.py", line 1280, in tokenize
    return 

In [67]:
bad_batch = note.with_row_index().with_columns(
    (pl.col('index') // 500).alias('batch')
).filter(
    pl.col('note').is_null()
)['batch'].to_list()

In [68]:
note = note.with_columns(
    pl.col('note').fill_null('')
)

In [58]:
import polars.selectors as cs

In [74]:
feat_retry = model.preprocess(
    data = {
        'icd': icd.rename({'bdsp_patient_id':'id'}).cast({'id':pl.Utf8, 'date':pl.Date}),
        'med': med.with_columns(
            pl.col('date').str.split(' ').list.first().str.to_date('%Y-%m-%d')
        ).with_columns(
            pl.coalesce(['date', 'date_start']).cast(pl.Date).alias('date')
        ).rename({'bdsp_patient_id':'id'}).cast({'id':pl.Utf8}),
        'note': note.rename({'bdsp_patient_id':'id'}).cast({'id':pl.Utf8, 'date':pl.Date}).with_row_index().with_columns(
            (pl.col('index') // 500).alias('batch')
        ).filter(
            pl.col('batch').is_in(bad_batch)
        ).drop('index'),
    },
    show_progress=True,
)

Processing narcolepsy notes: 100%|██████████| 6000/6000 [00:06<00:00, 994.10it/s]


In [85]:
feat_retry = note.rename({'bdsp_patient_id':'id'}).cast({'id':pl.Utf8, 'date':pl.Date}).with_row_index().with_columns(
    (pl.col('index') // 500).alias('batch')
).filter(
    pl.col('batch').is_in(bad_batch)
).select('index').hstack(feat_retry)

In [93]:
feat = pl.concat([feat.with_row_index().filter(
    ~pl.col('index').is_in(feat_retry['index'].to_list())
),
feat_retry]).drop('index').sort([pl.col('id').cast(pl.Int32), 'date'])

In [42]:
import subprocess
subprocess.run(['curl', '-d', 'hi', 'ntfy.sh/ejo3dJ7933'])

{"id":"lN8i0jUkTITY","time":1770318779,"expires":1770361979,"event":"message","topic":"ejo3dJ7933","message":"hi"}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   117  100   115  100     2   1837     31 --:--:-- --:--:-- --:--:--  1887


CompletedProcess(args=['curl', '-d', 'hi', 'ntfy.sh/ejo3dJ7933'], returncode=0)

In [96]:
control_df.write_parquet('/home/niels/Desktop/pheno_models/narcolepsy/control_update/controls.parquet')

In [99]:
note.group_by(['bdsp_patient_id', 'date']).agg(
    'filename'
).write_parquet('/home/niels/Desktop/pheno_models/narcolepsy/control_update/control_notes.parquet')

In [94]:
feat.write_parquet('/home/niels/Desktop/pheno_models/narcolepsy/control_update/feat.parquet')